# Hyperparameter Tuning with FLAML

|  | | | |
|-----|--------|--------|--------|
|![synapse](https://microsoft.github.io/SynapseML/img/logo.svg)| <img src="https://www.microsoft.com/en-us/research/uploads/prod/2020/02/flaml-1024x406.png" alt="drawing" width="200"/> | 


<style>
td, th {
   border: none!important;
}
</style>
In this notebook, we use FLAML to finetune a SynapseML LightGBM regression model for predicting house price. We use [*california_housing* dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html#sklearn.datasets.fetch_california_housing). The data consists of 20640 entries with 8 features.

The result shows that with **2 mins** of tuning, FLAML **improved** the metric R^2 **from 0.71 to 0.81**.

We will perform the task in following steps:
- **Setup** environment
- **Prepare** train and test datasets
- **Train** with initial parameters
- **Finetune** with FLAML
- **Check** results


## 1. Setup environment

In this step, we first install FLAML and MLFlow, then setup mlflow autologging to make sure we've the proper environment for the task. 

In [ ]:
%pip install "flaml[synapse]@https://automlsaeastus.blob.core.windows.net/releases/FLAML-latest-py3-none-any.whl" openml

## 2. Prepare train and test datasets
In this step, we first download the dataset with sklearn.datasets, then convert it into a spark dataframe. After that, we split the dataset into train, validation and test datasets.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

feature_cols = ["f" + str(i) for i in range(data.data.shape[1])]
header = ["target"] + feature_cols
df = spark.createDataFrame(
    pd.DataFrame(data=np.column_stack((data.target, data.data)), columns=header)
).repartition(1)

print("Dataframe has {} rows".format(df.count()))

Here, we split the datasets randomly.

In [ ]:
from pyspark.ml.feature import VectorAssembler

# Convert features into a single vector column
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = featurizer.transform(df)["target", "features"]

train_data, test_data = data.randomSplit([0.85, 0.15], seed=41)
train_data_sub, val_data_sub = train_data.randomSplit([0.85, 0.15], seed=41)

train_data.head()

## 3. Train with initial parameters
In this step, we prepare a train function which can accept different config of parameters. And we train a model with initial parameters.

In [ ]:
from synapse.ml.lightgbm import LightGBMRegressor
from pyspark.ml.evaluation import RegressionEvaluator

def train(alpha, learningRate, numLeaves, numIterations, train_data=train_data_sub, val_data=val_data_sub):
    """
    This train() function:
     - takes hyperparameters as inputs (for tuning later)
     - returns the R2 score on the validation dataset

    Wrapping code as a function makes it easier to reuse the code later for tuning.
    """

    lgr = LightGBMRegressor(
        objective="quantile",
        alpha=alpha,
        learningRate=learningRate,
        numLeaves=numLeaves,
        labelCol="target",
        numIterations=numIterations,
    )

    model = lgr.fit(train_data)

    # Define an evaluation metric and evaluate the model on the validation dataset.
    predictions = model.transform(val_data)
    evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="target", metricName="r2")
    eval_metric = evaluator.evaluate(predictions)

    return model, eval_metric

Here, we train a model with default parameters.

In [ ]:
init_model, init_eval_metric = train(alpha=0.2, learningRate=0.3, numLeaves=31, numIterations=100, train_data=train_data, val_data=test_data)
print("R2 of initial model on test dataset is: ", init_eval_metric)

## 4. Tune with FLAML

In this step, we configure the search space for hyperparameters, and use FLAML to tune the model over the parameters.

In [ ]:
import flaml
import time

# define the search space
params = {
    "alpha": flaml.tune.uniform(0, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

# define the tune function
def flaml_tune(config):
    _, metric = train(**config)
    return {"r2": metric}

Here, we optimize the hyperparameters with FLAML. We set the total tuning time to 120 seconds.

In [ ]:
analysis = flaml.tune.run(
    flaml_tune,
    params,
    time_budget_s=120,  # tuning in 120 seconds
    num_samples=100,
    metric="r2",
    mode="max",
    verbose=5,
    )

In [ ]:
flaml_config = analysis.best_config
print("Best config: ", flaml_config)

## 5. Check results
In this step, we retrain the model using the "best" hyperparameters on the full training dataset, and use the test dataset to compare evaluation metrics for the initial and "best" model.

In [ ]:
flaml_model, flaml_metric = train(train_data=train_data, val_data=test_data, **flaml_config)

print("On the test dataset, the initial (untuned) model achieved R^2: ", init_eval_metric)
print("On the test dataset, the final flaml (tuned) model achieved R^2: ", flaml_metric)